# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   


import sys
sys.path.append('../')
import helper_functions as hf

# initialize the optimizer

In [2]:
ax_client_init_file_name = 'optimizer/optimizer_00.json'
ax_client = hf.optimizer_init()
ax_client

[INFO 06-14 10:50:18] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-14 10:50:18] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='s1', parameter_type=INT, range=[0, 100]), RangeParameter(name='s2', parameter_type=INT, range=[0, 100]), RangeParameter(name='s3', parameter_type=INT, range=[0, 100]), RangeParameter(name='s4', parameter_type=INT, range=[0, 100]), RangeParameter(name='s5', parameter_type=INT, range=[0, 100]), RangeParameter(name='s6', parameter_type=INT, range=[0, 100]), RangeParameter(name='s7', parameter_type=INT, range=[0, 100]), RangeParameter(name='s8', parameter_type=INT, range=[0, 100]), RangeParameter(name='s9', parameter_type=INT, range=[0, 100]), RangeParameter(name='s10', parameter_type=INT, range=[0, 100]), RangeParameter(name='s11', parameter_type=INT,

AxClient(experiment=Experiment(drug_surfactant))

In [3]:
ax_client.save_to_json_file(ax_client_init_file_name)

[INFO 06-14 10:50:18] ax.service.ax_client: Saved JSON-serialized state of optimization to `optimizer/optimizer_00.json`.


# generate recommendations

In [4]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 0
Very Important: Please Confirm the Iteration Number is Iteration 0
Very Important: Please Confirm the Iteration Number is Iteration 0


In [5]:
df_design, ax_client = hf.run_optimizer(current_iteration=n, n_trials=6)

[INFO 06-14 10:50:19] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/modelbridge/cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 06-14 10:50:19] ax.service.ax_client: Generated new trial 0 with parameters {'s1': 47, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 's9': 85, 's10': 76, 's11': 89, 's12': 88, 'surfactant_conc': 81, 'drug_conc': 86} using model Sobol.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/modelbridge/cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 06-14 10:50:19] ax.service.ax_client: Generated new trial 1 with parameters {'s1': 93, 's2': 38, 's3': 50, 's4': 84, 's5': 50, 's6': 65, 's7': 78

# process results

In [6]:
df_conc, df_vol = hf.design_to_conc_to_vol (df_design)



In [7]:
hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well='F1', deepplat_well='F1')

✅ Successfully wrote to: protocol/otflex_0.py


In [8]:
df_absorbance = hf.process_absorbance(iteration=n)

In [9]:
results = hf.build_results(df_design, df_conc, df_absorbance)

In [10]:
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,0,47,59,49,31,96,8,9,35,85,76,89,88,40.5,21.50,0.0,0.0,12
1,1,93,38,50,84,50,65,78,53,26,20,36,5,14.5,0.25,0.0,0.0,12
2,2,51,77,13,24,19,94,27,92,2,28,16,30,30.0,13.25,0.0,0.0,12
3,3,8,20,88,72,73,36,60,21,62,75,58,63,6.0,9.50,0.0,0.0,12
4,4,22,93,75,52,54,24,49,77,22,5,7,56,35.5,4.75,0.0,0.0,12
5,5,68,10,25,0,1,56,64,10,63,97,66,38,9.5,24.25,NaN,NaN,11


In [11]:
norm_results = hf.normalize_data(results, 'normalize')

In [12]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,0,47,59,49,31,96,8,9,35,85,76,89,88,0.81,21.50,0.0,0.0,1.000000
1,1,93,38,50,84,50,65,78,53,26,20,36,5,0.29,0.25,0.0,0.0,1.000000
2,2,51,77,13,24,19,94,27,92,2,28,16,30,0.60,13.25,0.0,0.0,1.000000
3,3,8,20,88,72,73,36,60,21,62,75,58,63,0.12,9.50,0.0,0.0,1.000000
4,4,22,93,75,52,54,24,49,77,22,5,7,56,0.71,4.75,0.0,0.0,1.000000
5,5,68,10,25,0,1,56,64,10,63,97,66,38,0.19,24.25,NaN,NaN,0.916667


# load the results to the optimizer

In [13]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-14 10:50:26] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-14 10:50:26] ax.service.ax_client: Completed trial 0 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.81, None), 'complexity': (1.0, None)}.


[INFO 06-14 10:50:26] ax.service.ax_client: Completed trial 1 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.29, None), 'complexity': (1.0, None)}.
[INFO 06-14 10:50:26] ax.service.ax_client: Completed trial 2 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.6, None), 'complexity': (1.0, None)}.
[INFO 06-14 10:50:26] ax.service.ax_client: Completed trial 3 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.12, None), 'complexity': (1.0, None)}.
[INFO 06-14 10:50:26] ax.service.ax_client: Completed trial 4 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.71, None), 'complexity': (1.0, None)}.
[INFO 06-14 10:50:26] ax.service.ax_client: Completed trial 5 with data: {'micelle_drug_conc': (nan, None), 'surfactant_conc': (0.19, None), 'complexity': (0.916667, None)}.
[INFO 06-14 10:50:26] ax.service.ax_client: Saved JSON-serialized state of optimization to `optimizer/optimizer_0_loaded.json`.


AxClient(experiment=Experiment(drug_surfactant))